In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from datetime import date, timedelta, datetime

In [0]:
# -------------------------
# Parameters
# -------------------------
job_id = int(dbutils.widgets.get("job_id"))
start_date = date.fromisoformat(dbutils.widgets.get("start_date"))
end_date = date.fromisoformat(dbutils.widgets.get("end_date"))

In [0]:
# -------------------------
# Generate toy data
# -------------------------
rows = []
d = start_date
while d <= end_date:
    rows.append(
        (job_id, d.isoformat(), "toy_metric", 100.0)
    )
    d += timedelta(days=1)

df = spark.createDataFrame(
    rows,
    ["job_id", "business_date", "metric_name", "metric_value"]
).withColumn(
    "created_at", current_timestamp()
).withColumn(
    "business_date", col("business_date").cast("date")
)


In [0]:
# -------------------------
# Write Delta (authoritative)
# -------------------------
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .partitionBy("job_id")
      .saveAsTable("job_results")
)


In [0]:
# -------------------------
# Export Parquet snapshot for API
# -------------------------
export_path = (
    f"abfss://results@esgteamstorage.dfs.core.windows.net/"
    f"export/job_id={job_id}"
)

df.write.mode("overwrite").parquet(export_path)

print(f"Job {job_id} completed")
print(f"Exported parquet to {export_path}")